In [1]:
import pandas as pd
import numpy as np
import os
import warnings
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough, RunnableMap, RunnableLambda
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
import warnings
warnings.filterwarnings('ignore')

C:\Users\aa\anaconda3\envs\gpu_env\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [7]:
!pip show langchain_google_genai

Name: langchain-google-genai
Version: 3.0.3
Summary: An integration package connecting Google's genai package and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/google
Author: 
Author-email: 
License: MIT
Location: C:\Users\aa\anaconda3\Lib\site-packages
Requires: filetype, google-ai-generativelanguage, langchain-core, pydantic
Required-by: 


In [15]:
df = pd.read_csv("BankFAQs.csv")
df.head()

,Question,Answer,Class
0,Do I need to enter ‘#’ after keying in my Card...,Please listen to the recorded message and foll...,security
1,What details are required when I want to perfo...,"To perform a secure IVR transaction, you will ...",security
2,How should I get the IVR Password if I hold a...,An IVR password can be requested only from the...,security
3,How do I register my Mobile number for IVR Pas...,Please call our Customer Service Centre and en...,security
4,How can I obtain an IVR Password,By Sending SMS request: Send an SMS 'PWD<space...,security


In [21]:
count = df.applymap(lambda x: 'Rs. ' in str(x)).sum().sum()
print(f"Count of 'Rs.': {count}")

Count of 'Rs.': 0


In [19]:
df = df.applymap(lambda x: str(x).replace('Rs. ', 'SYP'))

In [27]:
count = df.applymap(lambda x: 'Rs.' in str(x)).sum().sum()
print(f"Count of 'Rs.': {count}")

Count of 'Rs.': 0


In [25]:
df = df.applymap(lambda x: str(x).replace('Rs.', 'SYP.'))

In [35]:
count = df.applymap(lambda x: 'HDFC' in str(x)).sum().sum()
print(f"Count of 'HDFC': {count}")

Count of 'HDFC': 0


In [33]:
df = df.applymap(lambda x: str(x).replace('HDFC', 'Venezia'))

In [43]:
count = df.applymap(lambda x: 'India' in str(x)).sum().sum()
print(f"Count of 'India': {count}")

Count of 'India': 0


In [41]:
df = df.applymap(lambda x: str(x).replace('India', 'Syria'))

In [49]:
count = df.applymap(lambda x: 'india' in str(x)).sum().sum()
print(f"Count of 'india': {count}")

Count of 'india': 0


In [47]:
df = df.applymap(lambda x: str(x).replace('india', 'syria'))

In [ ]:
translated_data = [
    {"Class": "Installation and Setup", "Question": "How do I install the app on an Android device?", "Answer": "Open Google Play Store, search for the app name, tap Install, and open it after installation."},
    {"Class": "Installation and Setup", "Question": "The app doesn't work after installation — what should I do?", "Answer": "Restart the device, ensure the OS is updated, and reinstall the app if the issue persists."},
    {"Class": "User Account", "Question": "How do I create a new account?", "Answer": "Tap Sign Up, enter your email or phone number, choose a password, and verify the code sent."},
    {"Class": "User Account", "Question": "I can't receive the activation code — what should I do?", "Answer": "Ensure the correct phone/email is entered, check the spam folder, and request the code again."},
    {"Class": "Account Recovery", "Question": "I forgot my password, how can I recover it?", "Answer": "Select 'Forgot Password', enter your email or phone, and follow the reset link sent to you."},
    {"Class": "Account Recovery", "Question": "My account was disabled, how can I restore it?", "Answer": "Contact customer support via the help form and provide usage reason and verification details."},
    {"Class": "Security and Privacy", "Question": "How do I enable two-factor authentication (2FA)?", "Answer": "Go to Account Settings > Security > 2FA, choose SMS or Authenticator App, and follow instructions."},
    {"Class": "Security and Privacy", "Question": "Is my personal data stored?", "Answer": "Yes, some data is stored as per the privacy policy; we commit to encrypting and protecting it."},
    {"Class": "Billing and Payments", "Question": "How do I add a payment method?", "Answer": "Go to Payment Settings > Add Card/PayPal, enter the details, and verify them."},
    {"Class": "Billing and Payments", "Question": "A charge was made but I didn't purchase anything — what should I do?", "Answer": "Check your purchase history, and if it's not yours, contact payment support immediately."},
    {"Class": "Performance and Errors", "Question": "The app is slow — how can I improve performance?", "Answer": "Close other apps, free up storage, update the app, or reinstall if needed."},
    {"Class": "Performance and Errors", "Question": "Error message 500 appeared — what does it mean?", "Answer": "It's a server error; try again later and contact support if it persists."},
    {"Class": "Features and Usage", "Question": "How do I change my profile picture?", "Answer": "Go to Profile Page > Edit > Change Picture, and choose from gallery or camera."},
    {"Class": "Features and Usage", "Question": "How do I export a report or request an invoice?", "Answer": "Go to Account Settings > Billing or Orders > Export/Request Invoice."},
    {"Class": "Notification Management", "Question": "How do I disable app notifications?", "Answer": "Go to app settings or system settings and turn off or adjust notification preferences."},
    {"Class": "Notification Management", "Question": "I'm not receiving payment notifications — what should I do?", "Answer": "Ensure notifications are enabled in device and app settings, and check internet and battery saver."},
    {"Class": "Backup and Restore", "Question": "How do I enable backup?", "Answer": "Go to Settings > Backup and Sync, enable cloud backup, and choose the appropriate account."},
    {"Class": "Backup and Restore", "Question": "Can I restore my data if I lose the device?", "Answer": "Yes, data can be restored from cloud backup after logging in with the same account."},
    {"Class": "Compatibility and Updates", "Question": "What are the minimum requirements to run the app?", "Answer": "System requirements (OS version, memory, storage) are listed on the app store page."},
    {"Class": "Compatibility and Updates", "Question": "How do I manually update the app?", "Answer": "Open the app store, search for the app, and tap Update."},
    {"Class": "Quick Actions", "Question": "What should I do if I forget my password?", "Answer": "Correct answer: Contact support."}
    {"Class": "Account Access",
    "Question": "How do I log in to the Venezia  banking app?",
    "Answer": "1- You must visit the nearest Venezia bank branch to open an account.n2- The branch will provide you with your account number and password.n3- Open the app and enter your account number and password to enjoy the best experience."}
,
{
  "Class": "Account Information",
  "Question": "How do I check my bank balance through the Venezia app?",
  "Answer": "1- You need to log in to the Venezia banking app.n2- Your bank account balance will appear at the top of the Home page."
},
{
  "Class": "Currency Exchange",
  "Question": "How do I check the global currency exchange rates against the Syrian Pound (SYP)?",
  "Answer": "1- You need to log in to the Venezia banking app.n2- The Home page is divided into three sections.n3- In the last section, 'Exchange rates vs Syrian Pound,' you will find all the currencies displayed by Venezia Bank."
},
{
  "Class": "Money Transfer",
  "Question": "How do I transfer money from my account to another account through the app?",
  "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Transfer page, then choose Normal Transfer, then send.n3- Enter the recipient's account number and the amount to transfer.n4- Press the send button, then confirm with fingerprint or password."
},
{
  "Class": "Transaction History",
  "Question": "How do I check the transfers I have received this month?",
  "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Transfer page, then select Request.n3- You will find all the incoming transfers there."
},
{
  "Class": "Money Transfer",
  "Question": "How do I see what transfers I sent this month through the app?",
  "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Transfers page.n3- Select Sent Records.n4- You will see all the transfers you have sent."
   },
   {
  "Class": "Money Transfer",
  "Question": "How do I see the monthly summary of sent and received transfers through the app?",
  "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Transfers page.n3- Select Monthly Summary.n4- You will find all the details."
  },
  {
    "Class": "Invoice Creation",
    "Question": "How can I create an invoice through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Payments page.n3- Select Generate QR.n4- Enter the required details.n5- Tap the Create QR button.n6- A QR code will appear containing all the details you entered."
  },
  {
    "Class": "Invoice Payment",
    "Question": "How can I pay an invoice through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Payments page.n3- Select Scan QR.n4- Scan the QR code that contains the invoice details.n5- The invoice details will appear.n6- If you want to pay, tap Payment and confirm with your fingerprint or password. If you do not want to pay, tap Cancel."
  },
  {
    "Class": "Payment Details",
    "Question": "How can I see in detail what I paid this month through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Payments page.n3- Select Payment Records.n4- You will see all the payments you have made this month."
  },
  {
    "Class": "Received Payments",
    "Question": "How can I see how many payments I received this month through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Payments page.n3- Select Requests.n4- You will see all the payments you received this month."
  },
  {
    "Class": "Monthly Summary",
    "Question": "How can I see the monthly summary of paid and received payments through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Payments page.n3- Select Monthly Summary.n4- You will find all the details."
  },
  {
    "Class": "Bill Payment - Self",
    "Question": "How can I pay a bill by myself through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Bills page.n3- Select unpaid Bills.n4- You will see all the bills you have not paid yet.n5- Tap the bill you want to pay.n6- The bill details will appear.n7- Tap the Pay now button.n8- Confirm with your fingerprint or password."
  },
  {
    "Class": "Bill Payment - Shared",
    "Question": "How can I pay a bill with cost sharing with others through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Bills page.n3- Select unpaid Bills.n4- You will see all the bills you have not paid yet.n5- Tap the bill you want to pay.n6- The bill details will appear.n7- Tap the Sharing payment button.n8- Select the participants.n9- Confirm the sharing by tapping the Confirm button.n10- Confirm with your fingerprint or password."
  },
  {
    "Class": "Paid Bills",
    "Question": "How can I see which bills I have paid this month through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Bills page.n3- Select Paid(Last Month).n4- The paid bills will appear.n5- Tap the bill whose details you want to see.n6- The details will appear."
  },
  {
    "Class": "Add Friend",
    "Question": "How can I add a friend to share a bill with through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Bills page.n3- Select Friends.n4- Tap the button at the bottom right corner.n5- Search for a friend by entering their account number and tapping Search.n6- The username you searched for will appear.n7- Tap the add button to add them."
  },
  {
    "Class": "Delete Friend",
    "Question": "How can I delete a friend through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Bills page.n3- Select Friends.n4- Tap the delete button inside the field of the friend you want to remove."
  },
  {
    "Class": "Shared Bills - Not Fully Paid",
    "Question": "How can I see the bills I shared where not all participants have paid yet through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Bills page.n3- Select Shared(Active).n4- The shared bills will appear.n5- Tap the bill whose details you want to see.n6- The details will appear, showing who has paid and who has not paid yet."
  },
  {
    "Class": "Shared Bills - Fully Paid",
    "Question": "How can I see the bills I shared where all participants have paid through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Bills page.n3- Select Shared(Completed).n4- The shared bills will appear.n5- Tap the bill whose details you want to see.n6- The details will appear, showing who has paid."
  },
  {
    "Class": "Loan Application",
    "Question": "How can I apply for a loan through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Loans page.n3- Select Loan Offers.n4- All available loans will appear.n5- Tap the loan you want to apply for.n6- The full loan details will appear.n7- Tap the Apply for this loan button.n8- Confirm with your fingerprint or password.n9- After a while, you will receive a notification to discuss the loan and enter your information."
  },
  {
    "Class": "Loan Status",
    "Question": "How can I know if the loan has been approved or rejected through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Loans page.n3- Select My Applications.n4- All the loans you have applied for will appear.n5- If the status is Pending, the loan is still being processed.n6- If the status is Approved, the bank has approved the loan.n7- If the status is Rejected, the bank has rejected the loan."
  },
  {
    "Class": "Loan Payment",
    "Question": "How can I see the loans I have and pay a specific loan through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Loans page.n3- Select My Loans.n4- You will see all the loans you have.n5- Tap the loan you want to pay.n6- The full details will appear, and at the bottom you will see the Pay installment button.n7- A page will appear with some details in addition to the amount you need to enter.n8- After entering the amount, tap Pay now.n9- Confirm with your fingerprint or password."
  },
{
  "Class": "Money Transfer",
  "Question": "How do I see what transfers I sent this month through the app?",
  "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Transfers page.n3- Select Sent Records.n4- You will see all the transfers you have sent."
   },
   {
  "Class": "Money Transfer",
  "Question": "How do I see the monthly summary of sent and received transfers through the app?",
  "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Transfers page.n3- Select Monthly Summary.n4- You will find all the details."
  },
  {
    "Class": "Invoice Creation",
    "Question": "How can I create an invoice through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Payments page.n3- Select Generate QR.n4- Enter the required details.n5- Tap the Create QR button.n6- A QR code will appear containing all the details you entered."
  },
  {
    "Class": "Invoice Payment",
    "Question": "How can I pay an invoice through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Payments page.n3- Select Scan QR.n4- Scan the QR code that contains the invoice details.n5- The invoice details will appear.n6- If you want to pay, tap Payment and confirm with your fingerprint or password. If you do not want to pay, tap Cancel."
  },
  {
    "Class": "Payment Details",
    "Question": "How can I see in detail what I paid this month through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Payments page.n3- Select Payment Records.n4- You will see all the payments you have made this month."
  },
  {
    "Class": "Received Payments",
    "Question": "How can I see how many payments I received this month through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Payments page.n3- Select Requests.n4- You will see all the payments you received this month."
  },
  {
    "Class": "Monthly Summary",
    "Question": "How can I see the monthly summary of paid and received payments through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Payments page.n3- Select Monthly Summary.n4- You will find all the details."
  },
  {
    "Class": "Bill Payment - Self",
    "Question": "How can I pay a bill by myself through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Bills page.n3- Select unpaid Bills.n4- You will see all the bills you have not paid yet.n5- Tap the bill you want to pay.n6- The bill details will appear.n7- Tap the Pay now button.n8- Confirm with your fingerprint or password."
  },
  {
    "Class": "Bill Payment - Shared",
    "Question": "How can I pay a bill with cost sharing with others through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Bills page.n3- Select unpaid Bills.n4- You will see all the bills you have not paid yet.n5- Tap the bill you want to pay.n6- The bill details will appear.n7- Tap the Sharing payment button.n8- Select the participants.n9- Confirm the sharing by tapping the Confirm button.n10- Confirm with your fingerprint or password."
  },
  {
    "Class": "Paid Bills",
    "Question": "How can I see which bills I have paid this month through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Bills page.n3- Select Paid(Last Month).n4- The paid bills will appear.n5- Tap the bill whose details you want to see.n6- The details will appear."
  },
  {
    "Class": "Add Friend",
    "Question": "How can I add a friend to share a bill with through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Bills page.n3- Select Friends.n4- Tap the button at the bottom right corner.n5- Search for a friend by entering their account number and tapping Search.n6- The username you searched for will appear.n7- Tap the add button to add them."
  },
  {
    "Class": "Delete Friend",
    "Question": "How can I delete a friend through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Bills page.n3- Select Friends.n4- Tap the delete button inside the field of the friend you want to remove."
  },
  {
    "Class": "Shared Bills - Not Fully Paid",
    "Question": "How can I see the bills I shared where not all participants have paid yet through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Bills page.n3- Select Shared(Active).n4- The shared bills will appear.n5- Tap the bill whose details you want to see.n6- The details will appear, showing who has paid and who has not paid yet."
  },
  {
    "Class": "Shared Bills - Fully Paid",
    "Question": "How can I see the bills I shared where all participants have paid through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Bills page.n3- Select Shared(Completed).n4- The shared bills will appear.n5- Tap the bill whose details you want to see.n6- The details will appear, showing who has paid."
  },
  {
    "Class": "Loan Application",
    "Question": "How can I apply for a loan through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Loans page.n3- Select Loan Offers.n4- All available loans will appear.n5- Tap the loan you want to apply for.n6- The full loan details will appear.n7- Tap the Apply for this loan button.n8- Confirm with your fingerprint or password.n9- After a while, you will receive a notification to discuss the loan and enter your information."
  },
  {
    "Class": "Loan Status",
    "Question": "How can I know if the loan has been approved or rejected through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Loans page.n3- Select My Applications.n4- All the loans you have applied for will appear.n5- If the status is Pending, the loan is still being processed.n6- If the status is Approved, the bank has approved the loan.n7- If the status is Rejected, the bank has rejected the loan."
  },
  {
    "Class": "Loan Payment",
    "Question": "How can I see the loans I have and pay a specific loan through the app?",
    "Answer": "1- You need to log in to the Venezia banking app.n2- Go to the Loans page.n3- Select My Loans.n4- You will see all the loans you have.n5- Tap the loan you want to pay.n6- The full details will appear, and at the bottom you will see the Pay installment button.n7- A page will appear with some details in addition to the amount you need to enter.n8- After entering the amount, tap Pay now.n9- Confirm with your fingerprint or password."
  },
 {
    "Class": "Money Transfer",
    "Question": "What transfers does the bank provide through the app?",
    "Answer": "1- Standard transfer from one account to another.\n2- Recurring transfer from one account to multiple accounts on a scheduled basis."
  },
  {
    "Class": "Money Transfer",
    "Question": "What is the standard transfer in the app?",
    "Answer": "A standard transfer is when the user selects an amount from their personal account and transfers it to another person’s account, either within the same bank or to another bank. The transfer fee is deducted from the sender’s account."
  },
  {
    "Class": "Money Transfer",
    "Question": "What is the recurring transfer in the app?",
    "Answer": "A recurring transfer is when the user selects an amount from their personal account and transfers it to another person’s account within the same bank at a specific interval. The transfer fee is deducted from the sender’s account at the moment each scheduled transfer is sent."
  },
  {
    "Class": "Money Transfer",
    "Question": "How do I pay a nearby person through the app?",
    "Answer": "You can pay using the QR feature, which allows creating an invoice and paying it by scanning the QR code."
  },
  {
    "Class": "Security",
    "Question": "What is OTP?",
    "Answer": "OTP (One-Time Password) is a temporary 6-digit password sent via an authenticator app (such as Google Authenticator). It is used as an extra security layer to confirm the account owner's identity when performing sensitive transactions. It is valid for a short time and expires after use. Never share your OTP with anyone."
  },
  {
    "Class": "Security",
    "Question": "Why is fingerprint used in the app?",
    "Answer": "Fingerprint authentication is used to verify the account owner's identity when performing payments or money transfers inside the app."
  },
  {
    "Class": "Security",
    "Question": "Why does the app stop after half an hour?",
    "Answer": "To increase security: the app automatically stops after 30 minutes to prevent unauthorized transactions if the device is stolen while the app is open."
  },
  {
    "Class": "Bill Payment",
    "Question": "What is the benefit of sharing bill payment with friends through the app?",
    "Answer": "This feature helps split the bill amount when multiple people share a single bill. For example, partners sharing an electricity bill under one person's name can divide the payment easily."
  },
  {
    "Class": "Loans",
    "Question": "When is the loan approved and how do I know after applying through the app?",
    "Answer": "After applying, you will receive a notification from the bank requesting you to visit any branch to provide your personal details. Based on that, your loan will be approved or rejected, and you will be notified of the final decision."
  },
  {
    "Class": "Money Transfer",
    "Question": "How do I know the recipient received the amount through the app?",
    "Answer": "The recipient will receive a notification showing the amount received and the sender’s account details."
  },
  {
    "Class": "Currency",
    "Question": "Are currency value predictions always correct?",
    "Answer": "No, predictions are based on past data and typical change periods but cannot be precise due to political, economic, or other unpredictable factors."
  },
  {
    "Class": "Fees",
    "Question": "What are the fee percentages deducted from amounts transferred and paid?",
    "Answer": "Fees vary according to the global financial system. To know the current fees, contact any branch via the app by pressing the side options button, selecting 'Contact us', and choosing the branch you want."
  },
  {
    "Class": "Branches",
    "Question": "Where is the bank’s main branch located?",
    "Answer": "The bank has branches worldwide. The main controlling branch is located in Venice, Italy."
  },
    {
  "Class": "Branches",
  "Question": "What branches exist in Syria?",
  "Answer": "Damascus Branches:\n1- Mazzeh (Phone: 0914837265)\n2- Al-Midan (Phone: 0947261835)\n3- Abu Rummaneh (Phone: 0935814726)\n4- Al-Malki (Phone: 0926748351)\n5- Kafr Souseh (Phone: 0951372846)\n\nDaraa Branches:\n1- Daraa al-Balad (Phone: 0945593648)\n2- Al-Mahatta (Phone: 0945593648)\n3- Al-Gizza (Phone: 0945593648) \n\n Homs Branches:\n1- Al-Hamra (Phone: 0984517263)\n2- Khaldiyeh (Phone: 0946271853)\n\nAleppo Branches:\n1- Al-Suleimaniyeh (Phone: 0927351846)\n2- Al-Aziziyeh (Phone: 0948612735)\n\nHama Branches:\n1- Al-Hader (Phone: 0917852463)\n2- Al-Arbaeen (Phone: 0964251738)\n\n Latakia Branches:\n 1- Al-Raml al-Janoubi (Phone: 0935162847)\n 2- Al-Azizieh (Phone: 0948271635)\n\n Tartus Branches:\n1- Al-Kornish (Phone: 0927164835)\n2- Al-Qadmus (Phone: 0943852716)\n\n Idlib Branches:\n1- Idlib City Center (Phone: 0937826154)\n2- Ariha (Phone: 0945163827)\n\n Raqqa Branches:\n1- Raqqa City Center (Phone: 0926481735)\n2- Al-Mansoura (Phone: 09417386251)\n\nAl-Hasakah Branches:\n1- Al-Hasakah City Center (Phone: 09361528471)\n\nDeir ez-Zor Branches:\n1- Al-Joura (Phone: 0927356184)"
},
    {
    "Class": "Currencies",
    "Question": "What currencies does the bank deal with?",
    "Answer": "The fixed supported currencies are:n- US Dollarn- Euron- UAE Dirhamn- Saudi Riyaln- Kuwaiti Dinarn- British Poundn- Japanese Yenn- Turkish Liran- Egyptian Poundn- Iraqi Dinar"
  },
  {
    "Class": "Currencies",
    "Question": "What is the red and green arrow next to currency prices?",
    "Answer": "The arrow shows today's price direction compared to the previous day: a red arrow down means the price decreased, a green arrow up means the price increased."
  },
  {
    "Class": "Customer Service",
    "Question": "How do I submit a complaint?",
    "Answer": "You can submit a complaint by contacting any branch through the 'Contact us' option in the app."
  },
  {
    "Class": "Customer Service",
    "Question": "What are the working hours and customer service hours?",
    "Answer": "Working hours are generally from 9 AM to 5 PM, varying according to the local time of your country."
  },
  {
    "Class": "Security",
    "Question": "I received a transaction I did not make—how do I stop it?",
    "Answer": "Contact any branch immediately to freeze your account and stop any further transactions until the issue is clarified."
  },
  {
    "Class": "Accounts",
    "Question": "How do I open a new account and what are the required conditions?",
    "Answer": "Visit the nearest branch with your personal details. Requirements vary by current regulations."
  },
  {
    "Class": "Accounts",
    "Question": "What is the minimum amount to open an account?",
    "Answer": "Minimum amounts vary by country. For USD accounts, the minimum is $200."
  },
  {
    "Class": "Accounts",
    "Question": "Can I create an account online?",
    "Answer": "No, to ensure accuracy of personal details, accounts must be opened in person at a branch."
  },
  {
    "Class": "Accounts",
    "Question": "How do I update my information?",
    "Answer": "Visit the nearest branch to update any personal information."
  },
  {
    "Class": "Accounts",
    "Question": "How do I get my account number?",
    "Answer": "You can find your account number inside the app under your Profile."
  },
  {
    "Class": "Accounts",
    "Question": "I forgot the account number or password—how do I change it?",
    "Answer": "Go to the nearest branch to receive your account details and set a new password for security."
  },
  {
    "Class": "Security",
    "Question": "What should I do if my phone is lost/stolen and the app was active?",
    "Answer": "Immediately contact any branch to freeze your account and prevent unauthorized transactions."
  },
  {
    "Class": "Transfers",
    "Question": "What are the daily/monthly transfer limits?",
    "Answer": "Limits vary by country and region. Contact any bank branch for accurate information."
  },
  {
    "Class": "Security",
    "Question": "I forgot my password—how do I recover it?",
    "Answer": "Visit the nearest branch to receive a new password to secure your account."
  },
  {
    "Class": "Technical Support",
    "Question": "The app is not working / it crashes—what is the solution?",
    "Answer": "Check your internet speed and region restrictions. If the account is frozen, contact any branch to resolve the issue."
  },
  {
    "Class": "Service Availability",
    "Question": "Does the service work outside the country?",
    "Answer": "Yes, in Iraq, Syria, Egypt, Turkey, Japan, UK, Kuwait, Saudi Arabia, UAE, European Union (Eurozone), and the United States."
  },
  {
    "Class": "Loans",
    "Question": "What types of loans are available?",
    "Answer": "The bank offers various loans with different durations and purposes. Check current loans in the app or contact a branch."
  },
  {
    "Class": "Loans",
    "Question": "What are the conditions to get a loan?",
    "Answer": "Age (Age): Must be within the bank’s accepted age range.\nIncome (Income): Sufficient, regular income that covers the monthly installment.\n Loan Amount (LoanAmount): Must match your income and current obligations.\n Credit score/record (CreditScore): The higher it is, the higher the approval chance and possibly a better interest rate.nEmployment stability (MonthsEmployed + EmploymentType): Longer employment and stable job type increase approval chances.\n Number of credit obligations (NumCreditLines): The number of existing loans is considered to assess commitment.\n Debt-to-income ratio (DTIRatio): Should not be high because it indicates financial pressure.\n Loan term (LoanTerm): Selected so that the installment fits your ability to pay.\n Loan purpose (LoanPurpose): Some purposes have different requirements or limits.\n Existing mortgage (HasMortgage): If you have a mortgage/home loan, it may affect eligibility due to higher obligations.\n Dependents and marital status (HasDependents + MaritalStatus): Used to estimate expenses and monthly commitments.\n Co-signer (HasCoSigner): May increase approval chances or improve terms.\n Interest rate (InterestRate): Determined based on risk (especially CreditScore, DTI, and employment stability)."
  },
  {
    "Class": "Loans",
    "Question": "How do I repay the loan early, and is there a penalty for late repayment?",
    "Answer": "You can repay at any time. There is a penalty for late repayment, and if the required amount is not paid, pledged assets may be seized by the bank."
  },
  {
    "Class": "Transactions",
    "Question": "Is there cash withdrawal and deposit?",
    "Answer": "Yes, of course—available at any branch and at any time, under conditions. For deposit: the person must have personal identification details, and it does not matter if they are the account owner. For withdrawal: the person must have personal identification details and must be the account owner of the account from which the withdrawal is made."
  }


]

# تحميل الملف أو إنشاء جديد
try:
    df = pd.read_csv('BankFAQs1.csv')
except FileNotFoundError:
    df = pd.DataFrame(columns=['Question', 'Answer', 'Class'])

# دمج البيانات
df = pd.concat([df, pd.DataFrame(translated_data)], ignore_index=True)

# حفظ الملف
df.to_csv('BankFAQs2.csv', index=False)
print("✅ All translated questions have been added and saved.")

In [3]:
bank = pd.read_csv("BankFAQs2.csv")
bank.head()

,Question,Answer,Class
0,Do I need to enter ‘#’ after keying in my Card...,Please listen to the recorded message and foll...,security
1,What details are required when I want to perfo...,"To perform a secure IVR transaction, you will ...",security
2,How should I get the IVR Password if I hold a...,An IVR password can be requested only from the...,security
3,How do I register my Mobile number for IVR Pas...,Please call our Customer Service Centre and en...,security
4,How can I obtain an IVR Password,By Sending SMS request: Send an SMS 'PWD<space...,security


In [55]:
bank["content"] = bank.apply(lambda row: f"Question: {row['Question']}\nAnswer: {row['Answer']}", axis=1)

In [57]:
bank.head()

,Question,Answer,Class,content
0,Do I need to enter ‘#’ after keying in my Card...,Please listen to the recorded message and foll...,security,Question: Do I need to enter ‘#’ after keying ...
1,What details are required when I want to perfo...,"To perform a secure IVR transaction, you will ...",security,Question: What details are required when I wan...
2,How should I get the IVR Password if I hold a...,An IVR password can be requested only from the...,security,Question: How should I get the IVR Password i...
3,How do I register my Mobile number for IVR Pas...,Please call our Customer Service Centre and en...,security,Question: How do I register my Mobile number f...
4,How can I obtain an IVR Password,By Sending SMS request: Send an SMS 'PWD<space...,security,Question: How can I obtain an IVR Password \nA...


In [61]:
documents = []
for _, row in bank.iterrows():

    documents.append(Document(page_content=row["content"], metadata={"class": row["Class"]}))
documents[1]

Document(metadata={'class': 'security'}, page_content='Question: What details are required when I want to perform a secure IVR transaction\nAnswer: To perform a secure IVR transaction, you will need your 16-digit Card number, Card expiry date, CVV number, mobile number and IVR password.')

# احفظ الموديل  في مجلد على جهازك

model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)
model.save("path_to_local_model")

In [7]:
hg_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# انشاء قاعده البيانات

persist_directory = './chroma_db3'
langchain_chroma = Chroma.from_documents(
    documents=documents,
    collection_name="chatbot_finance",
    embedding=hg_embeddings,
    persist_directory=persist_directory
)

# تحميل قاعدة البيانات

In [9]:
persist_directory = './chroma_db3'

langchain_chroma = Chroma(
    collection_name="chatbot_finance",
    embedding_function=hg_embeddings,
    persist_directory=persist_directory
)

# تجهيز القالب

In [12]:
template = PromptTemplate(
    input_variables=["question", "context"],
    template=(
"""
You are the official AI Finance Expert for "Venezia Bank". 

- Your primary identity is "Venezia Bank Assistant". 
- You specialize in banking services (accounts, loans, cards) AND general financial securities (stocks, bonds, investment funds).

STRICT RULES:
1. LOYALTY: You only represent Venezia Bank. If a user asks about another bank (e.g., Al Rajhi, HSBC, etc.), respond with: 
   "I am the dedicated assistant for Venezia Bank. I cannot provide information or comparisons regarding other banks."
2. DOMAIN: Only answer questions related to finance, banking, or financial securities.
3. OUT-OF-DOMAIN: For non-financial questions (geography, history, etc.), respond with: 
   "Sorry, I can only help with financial and banking-related questions."
4. GROUNDEDNESS: Use the provided Context to answer. If the answer is not in the context, and it's a general finance question, answer concisely based on Venezia Bank's professional standards. If you don't know, say: "Sorry, I don't know."
5. GREETINGS: Be friendly and mention Venezia Bank (e.g., "Hello! Welcome to Venezia Bank.").

Question: {question}  
Context: {context}  
Answer:
"""
    )
)

# تحميل نموذج جيميني

In [76]:
llm = ChatGoogleGenerativeAI(
    #model="gemini-2.5-flash",
    #model="gemini-2.5-flash-lite",
    model="gemini-robotics-er-1.5-preview",

    google_api_key="Enter your api",
    temperature=0.3
)

In [14]:
from langchain_ollama import ChatOllama

llm_gemma4 = ChatOllama(
    model="gemma4:e2b",
    temperature=0,
)
#llm_qwen = ChatOllama(model="qwen3:4b", temperature=0)

# انشاء المسترجع

In [17]:
retriever = langchain_chroma.as_retriever(search_kwargs={"k": 1})

# بناء سلسلة RAG

In [22]:
rag_chain = ({"context": retriever,"question":RunnablePassthrough()}
             | template
             | llm_gemma4
             | StrOutputParser())

# تقييمه من خلال سؤاله من الاسئله المخزنه لدينا 

In [29]:
q = rag_chain.invoke("What is the minimum amount to open an account?")
q

'Hello! Welcome to Venezia Bank.\n\nRegarding your question, the minimum amounts to open an account vary by country. For USD accounts, the minimum is $200.'

In [31]:
q = rag_chain.invoke("What branches exist in Syria?")
q

'Hello! Welcome to Venezia Bank.\n\nBased on the information available, here are the bank branches listed in Syria:\n\n**Damascus Branches:**\n1. Mazzeh (Phone: 0914837265)\n2. Al-Midan (Phone: 0947261835)\n3. Abu Rummaneh (Phone: 0935814726)\n4. Al-Malki (Phone: 0926748351)\n5. Kafr Souseh (Phone: 0951372846)\n\n**Daraa Branches:**\n1. Daraa al-Balad (Phone: 0945593648)\n2. Al-Mahatta (Phone: 0945593648)\n3. Al-Gizza (Phone: 0945593648)\n\n**Homs Branches:**\n1. Al-Hamra (Phone: 0984517263)\n2. Khaldiyeh (Phone: 0946271853)\n\n**Aleppo Branches:**\n1. Al-Suleimaniyeh (Phone: 0927351846)\n2. Al-Aziziyeh (Phone: 0948612735)\n\n**Hama Branches:**\n1. Al-Hader (Phone: 0917852463)\n2. Al-Arbaeen (Phone: 0964251738)\n\n**Latakia Branches:**\n1. Al-Raml al-Janoubi (Phone: 0935162847)\n2. Al-Azizieh (Phone: 0948271635)\n\n**Tartus Branches:**\n1. Al-Kornish (Phone: 0927164835)\n2. Al-Qadmus (Phone: 0943852716)\n\n**Idlib Branches:**\n1. Idlib City Center (Phone: 0937826154)\n2. Ariha (Phone: 

In [33]:
q = rag_chain.invoke("What currencies does the bank deal with?")
q

'Hello! Welcome to Venezia Bank.\n\nThe fixed supported currencies that the bank deals with are: US Dollar, Euro, UAE Dirham, Saudi Riyal, Kuwaiti Dinar, British Pound, Japanese Yen, Turkish Lira, Egyptian Pound, and Iraqi Dinar.'

In [35]:
q = rag_chain.invoke("How can I see which bills I have paid this month through the app?")
q

'Hello! Welcome to Venezia Bank.\n\nTo see which bills you have paid through the app, please follow these steps:\n\n1.  Log in to the Venezia banking app.\n2.  Go to the Bills page.\n3.  Select Paid (Last Month).\n4.  The paid bills will appear.\n5.  Tap the bill whose details you want to see.\n6.  The details will appear.'

In [37]:
q = rag_chain.invoke("ما هو معيار التحويل في التطبيق")
q

'أهلاً! مرحباً بك في بنك فينيتسيا (Venezia Bank).\n\nأنا متخصص في الخدمات المصرفية والأوراق المالية. للأسف، لا تتوفر لدي المعلومات المتعلقة بمعيار التحويل داخل التطبيق في السياق الحالي. يرجى مراجعة قسم المساعدة داخل التطبيق للحصول على هذه التفاصيل.'

In [39]:
q = rag_chain.invoke("مرحباو كيف حالك")
q

'Hello! Welcome to Venezia Bank. I am doing well, thank you for asking. How may I assist you with your banking or investment needs today?'

In [41]:
q = rag_chain.invoke("اين يقع البنك")
q

"Hello! Welcome to Venezia Bank.\n\nSorry, I don't know the location based on the information provided. I can only help with financial and banking-related questions."

In [43]:
q = rag_chain.invoke("Where are the location oif bank?")
q

'Hello! Welcome to Venezia Bank.\n\nThe bank has branches worldwide, and the main controlling branch is located in Venice, Italy.'

# تقييمه على 300 سؤال باجابه عشوائيه 

In [93]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer, util

BankTestGemni = pd.read_csv("BankTestGemni3R.csv")

merged = pd.merge(BankTestGemni, bank, on="Question", how="inner")

subset = merged.iloc[:300].copy()

# 1. BLEU Score
smooth = SmoothingFunction().method1
y_true = subset["Answer"].str.lower().str.split()
y_pred = subset["GeneratedAnswer"].str.lower().str.split()

bleu_scores = [
    sentence_bleu([ref], hyp, smoothing_function=smooth)
    for ref, hyp in zip(y_true, y_pred)
]
bleu_avg = sum(bleu_scores) / len(bleu_scores)

# 2. ROUGE Score
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
rouge_scores = [
    scorer.score(ref, pred)
    for ref, pred in zip(subset["Answer"], subset["GeneratedAnswer"])
]
rouge_avg = {
    "rouge-1": sum([s["rouge1"].fmeasure for s in rouge_scores]) / len(rouge_scores),
    "rouge-2": sum([s["rouge2"].fmeasure for s in rouge_scores]) / len(rouge_scores),
    "rouge-l": sum([s["rougeL"].fmeasure for s in rouge_scores]) / len(rouge_scores),
}

# 3. Semantic Similarity
model = SentenceTransformer('all-MiniLM-L6-v2')
emb_true = model.encode(subset["Answer"].tolist(), convert_to_tensor=True)
emb_pred = model.encode(subset["GeneratedAnswer"].tolist(), convert_to_tensor=True)
similarities = util.cos_sim(emb_true, emb_pred)
semantic_avg = similarities.diag().mean().item()

# عرض النتائج
print(f"BLEU (avg): {bleu_avg:.2%}")
print("ROUGE (avg):", rouge_avg)
print(f"Semantic Similarity (avg): {semantic_avg:.2%}")

BLEU (avg): 11.08%
ROUGE (avg): {'rouge-1': 0.5803767875990905, 'rouge-2': 0.31492005855042765, 'rouge-l': 0.4512154991740404}
Semantic Similarity (avg): 83.27%


# تقييمه على 300 سؤال باجابه من القاعده 

In [94]:
BankTestGemni = pd.read_csv("BankTestGemni3.csv")

merged = pd.merge(BankTestGemni, bank, on="Question", how="inner")


subset = merged.iloc[:300].copy()

# 1. BLEU Score
smooth = SmoothingFunction().method1
y_true = subset["Answer"].str.lower().str.split()
y_pred = subset["GeneratedAnswer"].str.lower().str.split()

bleu_scores = [
    sentence_bleu([ref], hyp, smoothing_function=smooth)
    for ref, hyp in zip(y_true, y_pred)
]
bleu_avg = sum(bleu_scores) / len(bleu_scores)

# 2. ROUGE Score
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
rouge_scores = [
    scorer.score(ref, pred)
    for ref, pred in zip(subset["Answer"], subset["GeneratedAnswer"])
]
rouge_avg = {
    "rouge-1": sum([s["rouge1"].fmeasure for s in rouge_scores]) / len(rouge_scores),
    "rouge-2": sum([s["rouge2"].fmeasure for s in rouge_scores]) / len(rouge_scores),
    "rouge-l": sum([s["rougeL"].fmeasure for s in rouge_scores]) / len(rouge_scores),
}

# 3. Semantic Similarity
model = SentenceTransformer('all-MiniLM-L6-v2')
emb_true = model.encode(subset["Answer"].tolist(), convert_to_tensor=True)
emb_pred = model.encode(subset["GeneratedAnswer"].tolist(), convert_to_tensor=True)
similarities = util.cos_sim(emb_true, emb_pred)
semantic_avg = similarities.diag().mean().item()

# عرض النتائج
print(f"BLEU (avg): {bleu_avg:.2%}")
print("ROUGE (avg):", rouge_avg)
print(f"Semantic Similarity (avg): {semantic_avg:.2%}")

BLEU (avg): 88.84%
ROUGE (avg): {'rouge-1': 0.9321506198990844, 'rouge-2': 0.911376805325945, 'rouge-l': 0.924128110096844}
Semantic Similarity (avg): 95.24%
